# Notebook 3: Equity Analysis — Who Pays Most, Gets Least?
Rewrites notebook 09 with actual charts instead of print statements.

Outputs: `equity_income_stratification.png`, `equity_cost_benefit_ratio.png`,
`equity_flight_subsidy.png`, `equity_gender_comparison.png`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

costs  = pd.read_csv('../data/processed/costs.csv')
schols = pd.read_csv('../data/processed/scholarship_outcomes.csv')
demo   = pd.read_csv('../data/processed/demographics.csv')

## 1. Income Stratification — Who Plays Club Soccer?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Income distribution of club sport families (Bocarro et al. 2018 / R002)
income_buckets = ['<$25k','$25–50k','$50–75k','$75–100k','$100–150k','$150k+']
club_pct   = [2, 6, 12, 20, 30, 30]   # % of elite club families
us_pop_pct = [12, 14, 14, 13, 20, 27] # % of US households (Census)

x = np.arange(len(income_buckets))
w = 0.35
axes[0].bar(x - w/2, us_pop_pct,  w, label='US Population',    color='#3498db', alpha=0.8)
axes[0].bar(x + w/2, club_pct,    w, label='Club Soccer Families', color='#e74c3c', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(income_buckets, rotation=20, ha='right')
axes[0].set_title('Income Distribution: Club Soccer Families vs US Population\n(Bocarro et al. 2018)', fontsize=11)
axes[0].set_ylabel('% of Group')
axes[0].legend()

# Cumulative exclusion bar
cutoffs = [25, 50, 75, 100]
us_below   = [12, 26, 40, 53]
club_below = [2,  8, 20, 40]
axes[1].plot(cutoffs, us_below,   'o-', color='#3498db', linewidth=2, label='US population below income')
axes[1].plot(cutoffs, club_below, 's-', color='#e74c3c', linewidth=2, label='Club soccer families below income')
axes[1].fill_between(cutoffs, us_below, club_below, alpha=0.15, color='#e74c3c',
                     label='Exclusion gap')
axes[1].set_title('Cumulative Income Exclusion from Club Soccer\n(Higher gap = more exclusion at that income level)', fontsize=11)
axes[1].set_xlabel('Household Income Threshold ($k)')
axes[1].set_ylabel('% of group below threshold')
axes[1].set_xticks(cutoffs)
axes[1].set_xticklabels(['<$25k','<$50k','<$75k','<$100k'])
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/analysis/equity_income_stratification.png', dpi=150)
plt.show()

print('Income stats:')
print('  60% of club soccer families earn $100k+ (vs 47% of US households)')
print('  Only 8% of club families earn <$50k (vs 26% of US households)')

## 2. Who Loses Most — Cost-to-Benefit Ratio by Group

In [ ]:
groups = [
    # (group_label, total_investment, expected_return, ratio_label)
    ('Competitive travel player\n(D2/D3 path)',              18_600,  45_000, '+2.4x GAIN'),
    ('Female ECNL player',                                   52_500,   8_000, '-6.5x loss'),
    ('Average elite (T3) player',                            56_000,   2_500, '-22x loss'),
    ('Flight 2/3 at large club\n(subsidizer)',               56_000,   2_000, '-28x loss'),
    ('Regional market elite\n(geographic bad luck)',         56_000,   1_000, '-56x loss'),
]

labels    = [g[0] for g in groups]
invest    = np.array([g[1] for g in groups])
returns   = np.array([g[2] for g in groups])
ratio_lbl = [g[3] for g in groups]
net       = returns - invest
bar_colors = ['#27ae60' if n >= 0 else '#e74c3c' for n in net]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(labels))
w = 0.3
ax.bar(x - w/2, invest/1e3,  w, label='Total Investment ($k)',  color='#2c3e50', alpha=0.85)
ax.bar(x + w/2, returns/1e3, w, label='Expected Return ($k)',   color='#3498db', alpha=0.85)

for i, (lbl, n) in enumerate(zip(ratio_lbl, net)):
    y_pos = max(invest[i], returns[i]) / 1e3 + 1
    ax.text(i, y_pos, lbl, ha='center', fontsize=8.5, fontweight='bold',
            color='#27ae60' if n >= 0 else '#c0392b')

ax.set_xticks(x)
ax.set_xticklabels(labels, ha='center', fontsize=9)
ax.set_title('Investment vs Expected Return by Player Group\n("Return" = expected scholarship + pro earnings, probability-weighted)', fontsize=12)
ax.set_ylabel('USD (thousands)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'${v:.0f}k'))
ax.legend()
plt.tight_layout()
plt.savefig('../data/analysis/equity_cost_benefit_ratio.png', dpi=150)
plt.show()

## 3. The Flight 2/3 Subsidy Structure

In [ ]:
# Typical large club structure: Flight 1 (top team) subsidized by lower flights
flight_data = pd.DataFrame({
    'Flight': ['Flight 1\n(top team)', 'Flight 2', 'Flight 3', 'Flight 4'],
    'Annual Fee': [8_000, 8_000, 7_500, 6_500],
    'Coaching Hours/Wk': [12, 8, 6, 5],
    'Field Quality': [90, 65, 50, 40],  # index: 100=best
    'Recruiter Visits': [25, 8, 3, 1],
    'Guest Player Displacement (%)': [0, 15, 25, 30],
})

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

x = np.arange(4)
flight_labels = flight_data['Flight'].values

# Fee vs coaching hours
axes[0].bar(x, flight_data['Annual Fee'], color='#2c3e50', alpha=0.85, label='Annual Fee')
ax2 = axes[0].twinx()
ax2.plot(x, flight_data['Coaching Hours/Wk'], 'o-', color='#e74c3c', linewidth=2, label='Coaching hrs/wk')
axes[0].set_xticks(x)
axes[0].set_xticklabels(flight_labels, fontsize=8)
axes[0].set_title('Same Fee, Less Coaching', fontsize=11)
axes[0].set_ylabel('Annual Fee (USD)')
ax2.set_ylabel('Coaching Hours/Week')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'${v:,.0f}'))

# Recruiter visits
axes[1].bar(x, flight_data['Recruiter Visits'], color='#8e44ad', alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(flight_labels, fontsize=8)
axes[1].set_title('College Recruiter Visits per Season', fontsize=11)
axes[1].set_ylabel('Visits (approx)')
for i, v in enumerate(flight_data['Recruiter Visits']):
    axes[1].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# Guest player displacement
axes[2].bar(x, flight_data['Guest Player Displacement (%)'], color='#e74c3c', alpha=0.85)
axes[2].set_xticks(x)
axes[2].set_xticklabels(flight_labels, fontsize=8)
axes[2].set_title('Roster Spots Lost to Guest Players (%)', fontsize=11)
axes[2].set_ylabel('% of roster spots')
for i, v in enumerate(flight_data['Guest Player Displacement (%)']):
    axes[2].text(i, v + 0.5, f'{v}%', ha='center', fontweight='bold')

plt.suptitle('The Flight 2/3 Subsidy Problem: Equal Pay, Unequal Product', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../data/analysis/equity_flight_subsidy.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Gender Inequity — Same Investment, Different Outcomes

In [ ]:
gender_data = pd.DataFrame({
    'Metric': [
        'D1 college programs',
        'D1 roster spots',
        'Pro league spots (1st div)',
        'Pro league median salary ($k)',
        'Full scholarship rate (%)',
    ],
    'Men': [206, 9_996, 850, 325, 15],
    'Women': [341, 9_960, 600, 75, 35],
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pro league comparison
cats = ['D1 programs', 'D1 roster spots', 'Pro spots (1st div)']
men_vals   = [206, 9996, 850]
women_vals = [341, 9960, 600]
x = np.arange(len(cats))
w = 0.35
axes[0].bar(x - w/2, men_vals,   w, color='#2980b9', alpha=0.85, label='Men')
axes[0].bar(x + w/2, women_vals, w, color='#e91e8c', alpha=0.85, label='Women')
axes[0].set_xticks(x)
axes[0].set_xticklabels(cats, rotation=15, ha='right')
axes[0].set_title('Roster Spots Available by Gender', fontsize=12)
axes[0].set_ylabel('Spots')
axes[0].legend()
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'{v:,.0f}'))

# Salary and scholarship comparison
sal_cats = ['Pro median salary\n($k/yr)', 'D1 full scholarship\nrate (%)', 'D1 avg scholarship\n($k/yr)']
sal_men   = [325, 15, 34]
sal_women = [75,  35, 40]
x2 = np.arange(len(sal_cats))
axes[1].bar(x2 - w/2, sal_men,   w, color='#2980b9', alpha=0.85, label='Men')
axes[1].bar(x2 + w/2, sal_women, w, color='#e91e8c', alpha=0.85, label='Women')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(sal_cats, rotation=10, ha='right')
axes[1].set_title('Salary & Scholarship Outcomes by Gender', fontsize=12)
axes[1].set_ylabel('Value')
axes[1].legend()

plt.suptitle('Gender Inequity in Youth Soccer Outcomes\nSimilar investment cost, very different pro salary ceiling', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('../data/analysis/equity_gender_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key gender finding:')
print('  Women have MORE D1 programs (341 vs 206) and HIGHER full scholarship rates (35% vs 15%)')
print('  But professional salary ceiling is 4.3x lower ($75k vs $325k median)')
print('  NWSL has 600 first-division spots vs MLS 850 — and earns 23% of MLS median')

## 5. LA City United — The Public Elite Benchmark

In [ ]:
# LA City United: $120/year vs private elite programs
comparison = pd.DataFrame({
    'Program': ['LA City United\n(Public)', 'Tier 3 Private\n(ECNL avg)', 'Tier 4 Private\n(MLS Next HD)'],
    'Annual Cost': [120, 7896, 22125],
    'Competitive Tier': ['T3/T4', 'T3', 'T4'],
    'MLS Draftees (est)': [2, 1, 3],
})

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(comparison['Program'], comparison['Annual Cost'],
              color=['#27ae60', '#e67e22', '#e74c3c'], alpha=0.85, edgecolor='white')
ax.set_title('Annual Cost: LA City United (Public) vs Private Elite Clubs', fontsize=13)
ax.set_ylabel('Annual Cost (USD)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'${v:,.0f}'))

for bar, cost in zip(bars, comparison['Annual Cost']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 150,
            f'${cost:,.0f}/yr', ha='center', fontsize=11, fontweight='bold')

# Annotate the ratio
ax.annotate('184x more\nexpensive', xy=(1, 7896), xytext=(0.5, 12000),
            arrowprops=dict(arrowstyle='->', color='black'),
            ha='center', fontsize=10, color='#c0392b')

plt.tight_layout()
plt.savefig('../data/analysis/equity_la_city_united.png', dpi=150)
plt.show()

premium_ratio = 7896 / 120
print(f'LA City United vs Tier 3 private club: {premium_ratio:.0f}x more expensive')
print(f'Both programs compete at the same tier against same opponents.')
print(f'The premium is for access, status, and brand — not measurably better development.')